In [1]:
import pandas as pd

X_train = pd.read_parquet('data/X_train.parquet')
y_train = pd.read_parquet('data/y_train.parquet').squeeze()  # Convertir a Series
X_val = pd.read_parquet('data/X_Val.parquet')
y_val = pd.read_parquet('data/y_Val.parquet').squeeze()  # Convertir a Series


In [2]:
X_train.drop(columns=['TransactionID', 'TransactionDT'], inplace=True)
X_val.drop(columns=['TransactionID', 'TransactionDT'], inplace=True)

print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

(472432, 403)
(472432,)
(118108, 403)
(118108,)


In [3]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f"Negativos: {neg}, Positivos: {pos}, scale_pos_weight: {scale:.2f}")

Negativos: 455833, Positivos: 16599, scale_pos_weight: 27.46


In [4]:
%pip install scikit-learn xgboost lightgbm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [12]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100,class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(scale_pos_weight=scale, random_state=42, n_jobs=-1,eval_metric='auc'),
    'LightGBM': lgb.LGBMClassifier(scale_pos_weight=scale, random_state=42, n_jobs=-1)
}



In [15]:
from sklearn.metrics import roc_auc_score, average_precision_score
import time

results = []

for name, model in models.items():
    print(f"Entrenando {name}")
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start_time
    print(f"Tiempo de entrenamiento: {elapsed:.2f} segundos")

    y_prob = model.predict_proba(X_val)[:, 1]

    auc = roc_auc_score(y_val, y_prob)
    pr_auc = average_precision_score(y_val, y_prob)

    results.append({'Modelo': name, 'ROC AUC': round(auc, 4), 'PR AUC': round(pr_auc, 4)})
    print(f" AUC: {auc:.4f} | PR AUC: {pr_auc:.4f}")

results_df = pd.DataFrame(results)
print("\nResultados comparativos:")
print(results_df)

Entrenando Random Forest
Tiempo de entrenamiento: 26.48 segundos
 AUC: 0.8855 | PR AUC: 0.5018
Entrenando XGBoost
Tiempo de entrenamiento: 3.95 segundos
 AUC: 0.8910 | PR AUC: 0.4948
Entrenando LightGBM
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.140353 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31976
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 401
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
Tiempo de entrenamiento: 4.05 segundos
 AUC: 0.8969 | PR AUC: 0.4854

Resultados comparativos:
          Modelo  ROC AUC  PR AUC
0  Random Forest   0.8855 

## **Selección del modelo ganador**

Entre la comparación de los 3 modelos, ganó LightGBM con 0.8969 en ROC-AUC, 
lo que indica una mejor capacidad para diferenciar entre clases. Por detrás 
queda Random Forest, que tuvo la puntuación más alta en PR-AUC (0.5018 vs 0.4854).

No se escogió Random Forest porque en un ambiente real, lo más importante para 
un banco es detectar cuáles transacciones son fraudes verdaderos, ya que son los 
casos que más problemas legales y económicos pueden traer. Para ello, LightGBM 
es más práctico: es 6 veces más rápido y lidera en ROC-AUC.

ROC-AUC mide qué tan bien el modelo diferencia entre clases en general, aunque 
puede inflarse por el desbalance de clases. PR-AUC muestra qué tan confiables 
son las predicciones de fraude y cuántos fraudes reales captura el modelo. 
LightGBM está casi a la par con Random Forest en PR-AUC, lo que confirma que 
no sacrifica detección real de fraudes a cambio de velocidad.

En conclusión, LightGBM ofrece las mejores métricas generales con el menor 
costo computacional. La diferencia en PR-AUC con Random Forest (0.016) no 
justifica un modelo 6 veces más lento. El siguiente notebook (`04_model_final.ipynb`) 
profundiza en LightGBM con tuning de hiperparámetros y análisis SHAP.